# 02 · Train the audio transformer: sound → EEG over 3 s, pretrained on public speech EEG

**Question.** Our videos are shown with sound. Sound onsets evoke strong, time-locked auditory responses (N1–P2 at ~100–200 ms), and the EEG follows the loudness envelope of speech with ~100–300 ms delay. Can a model that matches the **sound** of a video to the EEG over the whole 3 s identify the video? And does pretraining on ~120 h of public speech EEG help, as THINGS-EEG2 helps the video transformer (`01`)?

**Pretraining data.** The ICASSP 2023 Auditory EEG Decoding Challenge data (KU Leuven; a public part of SparrKULee, Accou et al., 2024; licence CC BY-NC 4.0): 71 people listening to Flemish audiobooks and podcasts, 64-channel EEG. We keep only **our 17 channels**, in our order, and cut the recordings into 3-s windows with their own sound, like our trials. Both datasets go through the same EEG preprocessing (band-pass 0.2–12 Hz, 50 Hz) and the same sound features.

**Model**: the time-resolved dual encoder of the team's CORTIVA pipeline (based on CORTIVA, Wang & Chen, 2026, arXiv:2608.01355), reduced to sound:

```
sound (150 steps of 20 ms × 145 features) → dropout → linear → 4 causal dilated conv blocks (receptive field 1.2 s:
                                            the latency is learned) → 32 latent time courses × 150
EEG (17 channels × 150 steps)             → per-subject + shared spatial projection 17→32 → temporal conv → 32 × 150
                                            [+ 2 transformer layers over time, whose output starts at zero]
score = mean correlation of the 32 latent time courses; trained with symmetric multi-positive InfoNCE
```

Sound features, computed by the same code for speech and videos: loudness envelope (1), Whisper's log-mel spectrum (80) and one Whisper-small encoder layer reduced by PCA (64).

**Variants**: the four of `01`, each with two **EEG encoders**: `cortiva` (convolutions only) and `cortiva+transformer` (two attention layers over time). Pretraining on many hours might make the larger encoder usable.

| variant | trained on our data |
|---|---|
| `scratch` | everything, from random weights (no speech EEG) |
| `zero-shot` | nothing: the speech model as it is (our subjects use its shared spatial projection) |
| `frozen` | only the spatial projections of our six subjects |
| `finetune` | everything, starting from the speech weights, with a 3× smaller learning rate |

**Validation and data rules** as in `01`: 5 folds × 102 held-out training videos, 4 seeds, all six subjects jointly, early stopping on 10 % of the fitting videos, all statistics from the fitting recordings. **No test EEG is read here** (`04_evaluate_audio_transformer` does that). The sound features of all 1,102 videos are computed here (stimulus features only; their scaling is fitted on training videos) and saved for `04`. 34 videos have no audio track; they get silence, which is what the subjects heard.

**Saved** to `~/eeg-results/pretrained-transformers/audio_transformer/<run>/` (layout as in `01`, plus `features/`). Downloads and the speech feature cache: `~/eeg-data/auditory-eeg-icassp2023/`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from math import gcd
from pathlib import Path
import gzip, hashlib, io, json, math, os, pickle, queue, re, sys, threading, urllib.error, urllib.request, zipfile

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')   # Hugging Face's widget bars freeze at 0 % here

import av
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from scipy.signal import resample_poly
from tqdm import tqdm                # text bars: widget bars (tqdm.auto) freeze at 0 % through the cluster's kernel gateway
from transformers import WhisperFeatureExtractor, WhisperModel

transformers.utils.logging.disable_progress_bar()

## 1. Configuration

In [ ]:
# --- Paths (GPU server) ------------------------------------------------------------------------------------------------
DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', str(Path.home() / 'eeg-data' / 'eeg-mind-reading-challenge'))).expanduser()
if (DATA_ROOT / 'eeg-mind-reading-challenge').is_dir():
    DATA_ROOT /= 'eeg-mind-reading-challenge'
AUDEEG_ROOT = Path(os.environ.get('AUDEEG_ROOT', str(Path.home() / 'eeg-data' / 'auditory-eeg-icassp2023'))).expanduser()
SPLIT_FILE = Path.cwd().resolve() / 'video_splits.json'      # team-wide folds (the GPU kernel's cwd is the cluster home)
RESULTS_ROOT = Path(os.environ.get('EEG_RESULTS_ROOT', str(Path.home() / 'eeg-results'))).expanduser()
OUT_ROOT = RESULTS_ROOT / 'pretrained-transformers' / 'audio_transformer'
RUN_ID = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUT_DIR = OUT_ROOT / RUN_ID

# --- Public speech EEG: ICASSP 2023 Auditory EEG Decoding Challenge (part of SparrKULee), CC BY-NC 4.0 ---------------------
AUDEEG_URL = 'https://homes.esat.kuleuven.be/~spchdata/corpora/auditory_eeg_data/ICASSP-2023-eeg-decoding-challenge-dataset/'
AUDEEG_FILES = ['preprocessed_eeg.zip',      # ~6 GB: EEG per recording (.pkl with 'eeg' and 'stimulus'), 64 Hz, 64 ch
                'stimuli.zip',               # ~3 GB: original audio (.npz(.gz) with 'audio' and 'fs')
                'preprocessed_stimuli.zip']  # ~0.3 GB: the challenge's envelopes, only for the channel check
AUDEEG_SFREQ = 64                    # dataset fact: the preprocessed EEG is at 64 Hz
# Standard order of a 64-channel BioSemi cap (A1-A32, B1-B32). The files carry no channel names, so this order is an
# assumption; section 3 checks it with the speech-tracking topography.
BIOSEMI64 = ['Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C1', 'C3', 'C5', 'T7', 'TP7',
             'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9', 'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz',
             'Fpz', 'Fp2', 'AF8', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz',
             'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'P8', 'P10', 'PO8', 'PO4', 'O2']

# --- Our data -------------------------------------------------------------------------------------------------------------
CHANNELS = ['Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2']
                                     # organisers (2026-10-18): the same channels and order for all subjects
CHANNEL_IDX = [BIOSEMI64.index(c) for c in CHANNELS]       # where our channels sit in the 64-channel cap
SUBJECTS = [1, 2, 3, 4, 5, 6]        # trained jointly: the training EEG of all six subjects may be used
TRAIN_VIDEO_IDS = np.arange(1, 1001)
N_CANDIDATES = 102                   # test candidates per subject; every validation fold has the same size
EEG_SFREQ = 500
N_FOLDS, SPLIT_SEED = 5, 0           # team recipe (spectral_model/01), only used if the split file does not exist yet

# --- Common format (both datasets) ------------------------------------------------------------------------------------------
BAND_HZ = (0.2, 12.0)                # EEG band of CORTIVA: 0.2 Hz removes drifts; little stimulus-locked signal above 12 Hz
RATE_HZ = 50                         # > 2 x 12 Hz (Nyquist), and the 20-ms step of Whisper's encoder and CORTIVA
WINDOW_S = 3.0                       # = our trials: 3 s from the onset of the sound
N_STEPS = round(WINDOW_S * RATE_HZ)  # 150 time steps per window
VAL_FRACTION = 0.1                   # pretraining validation: the last 10 % of every speech stimulus (the challenge's
                                     # rule; contiguous, so no window overlaps the training part)

# --- Sound features (identical code for both datasets) ------------------------------------------------------------------------
AUDIO_RATE = 16000                   # Whisper's input rate
ENVELOPE_POWER = 0.6                 # power-law compression of the envelope, as SparrKULee's spectrogram features
WHISPER_CHECKPOINT = 'openai/whisper-small'
WHISPER_LAYER = 12                   # best Whisper-small layer for our EEG in the layer-wise encoding analysis
                                     # (pretrained_transformers/02: validation top-5 0.104, chance 0.049)
WHISPER_PCA = 64                     # dims kept from Whisper, as CORTIVA (64 per layer)
MEL_STEPS_PER_STEP = 2               # Whisper's log-mel has a 10-ms hop: 2 frames per 20-ms step

# --- Model (CORTIVA handoff; we do not have its code, so details may differ) ----------------------------------------------------
HIDDEN, LATENT = 128, 32             # stimulus encoder width and the shared latent channels
KERNEL, DILATIONS = 5, (1, 2, 4, 8)  # causal dilated convolutions: receptive field 1 + 4 * (1+2+4+8) = 61 steps = 1.2 s
INPUT_DROPOUT = DROPOUT = 0.5        # CORTIVA: strong dropout against overfitting the ~5,650 recordings
GROUP_NORM_GROUPS = 8                # not specified in the handoff: 8 groups of 16 channels
EEG_KERNEL = 9                       # depthwise temporal convolution on the EEG side (180 ms)
TF_DIM, TF_LAYERS, TF_HEADS = 64, 2, 4   # transformer encoder: small (head size 16); its output starts at zero
TF_DROPOUT = 0.1                     # standard transformer dropout
EEG_ENCODERS = ['cortiva', 'cortiva+transformer']
TEMPERATURE_INIT = 0.07              # CLIP's initial softmax temperature (learned afterwards)

# --- Training -------------------------------------------------------------------------------------------------------------------
LR, WEIGHT_DECAY = 2e-3, 0.05        # CORTIVA
STIMULI_PER_BATCH = 16               # CORTIVA: 16 stimuli (videos or speech windows) plus all their recordings per batch
FT_LR = LR / 3                       # finetune: smaller steps keep the pretrained features (as in 01)
FT_MAX_EPOCHS = 60                   # 4x CORTIVA's stop epoch (15): the smaller finetune steps need more epochs
FT_PATIENCE = 6                      # 10 % of the budget, as in 01
INNER_FRACTION, INNER_SEED = 0.1, 0  # early stopping on 10 % of the fitting videos, the same for all variants (paired)
PRETRAIN_MAX_EPOCHS, PRETRAIN_PATIENCE = 20, 3   # stop when validation has not improved for 3 epochs
SEEDS = [0, 1, 2, 3]                 # averaged (row z-score, as CORTIVA), never selected; as in 01
VARIANTS = ['scratch', 'zero-shot', 'frozen', 'finetune']
SETTINGS = [(v, e) for v in VARIANTS for e in EEG_ENCODERS]
SELECTION_METRIC = 'top5'            # chooses the final setting: 5x more hits than top-1, so less noisy

# --- Compute ---------------------------------------------------------------------------------------------------------------------
N_GPUS = torch.cuda.device_count()
FT_JOBS_PER_GPU = 3                  # technical: small fine-tuning jobs do not fill a GPU
FEATURE_BATCH = 64                   # technical: 3-s windows per Whisper call
LOAD_THREADS = 16                    # technical: CPU threads for unzipping, resampling and decoding audio
assert N_GPUS > 0, 'run this notebook on the GPU kernel'

for d in [AUDEEG_ROOT / 'downloads', OUT_DIR / 'validation' / 'similarity', OUT_DIR / 'models', OUT_DIR / 'figures',
          OUT_DIR / 'features']:
    d.mkdir(parents=True, exist_ok=True)
print('public data ->', AUDEEG_ROOT, '\nresults ->', OUT_DIR)

### Helpers: validation folds, metrics, parallel GPU jobs (as in `01`)

In [ ]:
if SPLIT_FILE.is_file():
    FOLDS = [np.array(f) for f in json.loads(SPLIT_FILE.read_text())['validation_folds']]
else:                                # same recipe as the other team notebooks
    perm = np.random.default_rng(SPLIT_SEED).permutation(TRAIN_VIDEO_IDS)
    FOLDS = [np.sort(perm[i * N_CANDIDATES:(i + 1) * N_CANDIDATES]) for i in range(N_FOLDS)]
    SPLIT_FILE.write_text(json.dumps(dict(seed=SPLIT_SEED, note='validation videos per fold; the other training videos '
                                          'of a fold are used for fitting', validation_folds=[f.tolist() for f in FOLDS]),
                                     indent=1))
assert len(FOLDS) == N_FOLDS and all(len(f) == N_CANDIDATES for f in FOLDS)


def metrics(S, true_cols):
    """S: similarities (recordings, candidates). The rank of the true video is its place in the ranked list a
    submission would contain (stable sort). Silent videos all get the same score; this way such ties are never
    broken in the model's favour."""
    order = np.argsort(-np.asarray(S), axis=1, kind='stable')
    rank = np.argmax(order == np.asarray(true_cols)[:, None], axis=1) + 1
    return dict(n=len(rank), top1=float(np.mean(rank == 1)), top5=float(np.mean(rank <= 5)), mean_rank=float(rank.mean()))


def row_z(S):
    """Standardise every row (one recording's scores over the candidates) before averaging seeds (CORTIVA)."""
    return (S - S.mean(axis=1, keepdims=True)) / S.std(axis=1, keepdims=True)


def slug(setting):
    return re.sub('[^a-z0-9]+', '-', ' '.join(setting).lower()).strip('-')


def tag(cfg):
    """Short hash of a settings dict: caches and checkpoints are reused only with identical settings."""
    return hashlib.sha1(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:10]


STOP = threading.Event()             # set on interrupt or error: running trainings end at their next step


class Progress:
    """One shared text bar for parallel jobs (thread-safe): expect(n) adds n units to the total, tick(n) marks n units
    done, note(key, text) shows a job's status (epoch, scores) after the bar."""

    def __init__(self, bar):
        self.bar, self.lock, self.notes = bar, threading.Lock(), {}

    def expect(self, n):
        with self.lock:
            self.bar.total = (self.bar.total or 0) + n
            self.bar.refresh()

    def tick(self, n=1):
        with self.lock:
            self.bar.update(n)

    def note(self, key, text):
        with self.lock:
            self.notes[key] = text
            self.bar.set_postfix_str(' | '.join(f'{k}: {v}' for k, v in sorted(self.notes.items())), refresh=False)


def run_parallel(fn, items, jobs_per_gpu, desc, unit='run', count_items=True):
    """fn(item, progress, device) for all items, jobs_per_gpu at a time on every GPU; results in input order.
    count_items=True: one bar unit per finished item; False: the jobs report their own units (training steps)."""
    STOP.clear()
    for g in range(N_GPUS):          # CUDA's linear-algebra library loads lazily and not thread-safely: load it here
        torch.linalg.eigh(torch.eye(2, dtype=torch.float64, device=f'cuda:{g}'))
    devices = queue.Queue()
    for _ in range(jobs_per_gpu):
        for g in range(N_GPUS):
            devices.put(f'cuda:{g}')
    bar = tqdm(total=len(items) if count_items else 0, desc=desc, unit=unit)
    progress = Progress(bar)

    def on_free_gpu(item):
        device = devices.get()
        try:
            return fn(item, progress, device)
        finally:
            devices.put(device)

    out = [None] * len(items)
    pool = ThreadPoolExecutor(N_GPUS * jobs_per_gpu)
    try:                             # shuffled start order: slow and fast jobs mix, so the ETA stays steady
        futures = {pool.submit(on_free_gpu, items[i]): i for i in np.random.default_rng(0).permutation(len(items))}
        for fut in as_completed(futures):
            out[futures[fut]] = fut.result()
            if count_items:
                progress.tick()
    except BaseException:
        STOP.set()
        pool.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        bar.close()
    pool.shutdown()
    return out

## 2. Download the public speech EEG (CC BY-NC 4.0)

The server publishes no checksums. Each file is downloaded to `*.part`, its size is compared with the server's `Content-Length`, the zip is tested (CRC of every member), and only then is it renamed. Files already present are skipped.

In [ ]:
def remote_size(url):
    for method in ['HEAD', 'GET']:                       # some servers refuse HEAD; for GET only the headers are read
        try:
            with urllib.request.urlopen(urllib.request.Request(url, method=method)) as r:
                return int(r.headers['Content-Length'])
        except urllib.error.HTTPError:
            continue
    raise RuntimeError(f'no size for {url}')


def download(name, bar, lock):
    target = AUDEEG_ROOT / 'downloads' / name
    size = remote_size(AUDEEG_URL + name)
    if target.is_file() and target.stat().st_size == size:
        with lock:
            bar.update(size)
        return f'{name}: present'
    partial, seen = target.with_name(name + '.part'), [0]

    def hook(blocks, block_size, _):
        now = min(blocks * block_size, size)
        with lock:
            bar.update(now - seen[0])
        seen[0] = now

    urllib.request.urlretrieve(AUDEEG_URL + name, partial, reporthook=hook)
    assert partial.stat().st_size == size, f'{name}: incomplete download; delete {partial} and rerun'
    with zipfile.ZipFile(partial) as z:
        assert z.testzip() is None, f'{name}: damaged zip; delete {partial} and rerun'
    partial.rename(target)
    return f'{name}: {size / 1e9:.2f} GB, zip intact'


def extract(name):
    marker = AUDEEG_ROOT / f'.extracted_{name}'
    if not marker.is_file():
        with zipfile.ZipFile(AUDEEG_ROOT / 'downloads' / name) as z:
            z.extractall(AUDEEG_ROOT / name.replace('.zip', ''))
        marker.touch()


lock = threading.Lock()
bar = tqdm(total=sum(remote_size(AUDEEG_URL + n) for n in AUDEEG_FILES), desc='Download', unit='B', unit_scale=True)
with ThreadPoolExecutor(len(AUDEEG_FILES)) as pool:
    messages = list(pool.map(lambda n: download(n, bar, lock), AUDEEG_FILES))
bar.close()
print('\n'.join(messages))
with ThreadPoolExecutor(len(AUDEEG_FILES)) as pool:
    list(tqdm(pool.map(extract, AUDEEG_FILES), total=len(AUDEEG_FILES), desc='Unzip', unit='zip'))

## 3. Public EEG: load, check the channel order, keep our 17 channels

- **Loading.** The EEG files are Python pickles, and a pickle can run arbitrary code. `safe_load` lets only numpy arrays and plain data through.
- **Channel check.** The files carry no channel names. Physiology gives a test: auditory cortex lies in the temporal lobes, so with this dataset's average reference the correlation of the EEG with the speech envelope (0–300 ms later) should peak **bilaterally over temporal and fronto-central channels**. A wrong channel order would scatter the top channels over the head. The same figure shows how much speech tracking our 17 posterior channels keep.
- **Preprocessing** to the common format: FFT band-pass 0.2–12 Hz and resampling to 50 Hz in one step, then a z-score per channel and recording. The reference differs from ours (unknown); a reference change is a fixed linear mix of the channels, which the per-subject spatial projection can absorb.

In [ ]:
SAFE_GLOBALS = {('numpy.core.multiarray', '_reconstruct'), ('numpy._core.multiarray', '_reconstruct'),
                ('numpy.core.multiarray', 'scalar'), ('numpy._core.multiarray', 'scalar'),
                ('numpy', 'ndarray'), ('numpy', 'dtype'), ('collections', 'OrderedDict'),
                ('pathlib', 'PosixPath'), ('pathlib', 'PurePosixPath'), ('pathlib', 'WindowsPath')}


class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) in SAFE_GLOBALS:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f'refused to load {module}.{name}: only numpy arrays and plain data are allowed')


def safe_load(path):
    with open(path, 'rb') as f:
        return SafeUnpickler(f).load()


def bandpass_resample(x, sfreq, device='cuda:0'):
    """(channels, samples) at sfreq -> float32 (channels, samples') at RATE_HZ, band-passed to BAND_HZ. One FFT step:
    the recording is mirrored (no jump at the ends), frequencies outside the band are set to zero, and the inverse FFT
    is taken at the new length. The recording is first cut by < 1 s to a length that converts to whole samples."""
    step = sfreq // gcd(sfreq, RATE_HZ)
    x = torch.as_tensor(np.asarray(x)[:, :x.shape[1] // step * step], dtype=torch.float64, device=device)
    n = x.shape[1]
    m = n * RATE_HZ // sfreq
    spec = torch.fft.rfft(torch.cat([x, x.flip(-1)], dim=-1), dim=-1, norm='forward')
    f = torch.fft.rfftfreq(2 * n, d=1 / sfreq, dtype=torch.float64, device=device)
    spec[..., (f < BAND_HZ[0]) | (f > BAND_HZ[1])] = 0
    return torch.fft.irfft(spec[..., :m + 1], n=2 * m, dim=-1, norm='forward')[..., :m].float().cpu().numpy()


def load_recording(path):
    d = safe_load(path)
    eeg = np.asarray(d['eeg'], dtype=np.float32)
    eeg = eeg.T if eeg.shape[1] == len(BIOSEMI64) else eeg                  # -> (64 channels, time)
    assert eeg.shape[0] == len(BIOSEMI64), f'{path.name}: expected 64 channels, got shape {eeg.shape}'
    return path.name.split('_')[0], str(d['stimulus']).split('.')[0], eeg   # subject, stimulus name, EEG


files = sorted((AUDEEG_ROOT / 'preprocessed_eeg').rglob('sub-*.pkl'))
assert files, 'no preprocessed EEG found: run section 2'
RECORDINGS = [load_recording(p) for p in tqdm(files, desc='Loading public EEG', unit='recording')]
AUD_SUBJECTS = sorted({s for s, _, _ in RECORDINGS})
print(f'{len(RECORDINGS)} recordings, {len(AUD_SUBJECTS)} subjects, {len({n for _, n, _ in RECORDINGS})} stimuli, '
      f'{sum(e.shape[1] for _, _, e in RECORDINGS) / AUDEEG_SFREQ / 3600:.0f} h of EEG')

In [ ]:
# Channel check: speech tracking of all 64 channels, with the challenge's own envelopes (64 Hz, aligned at 0)
LAGS = np.arange(0, int(0.3 * AUDEEG_SFREQ) + 1)        # 0-300 ms: the latency range of auditory responses
CHECK_RECORDINGS = 60                                    # enough for a stable map (one per 2 h of EEG)
env_dir = AUDEEG_ROOT / 'preprocessed_stimuli'
track, used = torch.zeros(len(BIOSEMI64), len(LAGS), dtype=torch.float64, device='cuda:0'), 0
for subject, stim, eeg in tqdm(RECORDINGS[:CHECK_RECORDINGS], desc='Speech tracking per channel', unit='recording'):
    hits = list(env_dir.rglob(f'{stim}_envelope.npy'))
    if not hits:
        continue
    env = torch.as_tensor(np.load(hits[0]).reshape(-1), dtype=torch.float64, device='cuda:0')
    x = torch.as_tensor(eeg, dtype=torch.float64, device='cuda:0')
    n = min(x.shape[1], len(env)) - LAGS[-1]
    e = (env[:n] - env[:n].mean()) / env[:n].std()
    for j, lag in enumerate(LAGS):                       # the EEG lags behind the sound
        seg = x[:, lag:lag + n]
        track[:, j] += ((seg - seg.mean(1, keepdim=True)) / seg.std(1, keepdim=True)) @ e / n
    used += 1
TRACK = (track / max(used, 1)).abs().max(dim=1).values.cpu().numpy()
top = np.argsort(-TRACK)[:10]
print(f'{used} recordings; strongest speech tracking:', [f'{BIOSEMI64[i]} {TRACK[i]:.3f}' for i in top])
print(f'our 17 channels keep {TRACK[CHANNEL_IDX].mean() / TRACK[top].mean():.0%} of the tracking of the top-10 channels')

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(BIOSEMI64)), TRACK, color=['C3' if i in CHANNEL_IDX else 'C0' for i in range(len(BIOSEMI64))])
ax.set_xticks(range(len(BIOSEMI64)))
ax.set_xticklabels(BIOSEMI64, rotation=90, fontsize=7)
ax.set_ylabel('max |r| envelope vs EEG\n(lag 0-300 ms)')
ax.set_title('Speech tracking per channel in the public data (red = our 17 channels)', fontsize=10)
fig.savefig(OUT_DIR / 'figures' / 'speech_tracking_per_channel.png', dpi=130, bbox_inches='tight')
plt.show()

**Reference result** (`pretrained_transformers/06`, 2026-10-18): the top channels were T7, TP7, T8, FT8 and FT7, bilateral and temporal, so the assumed BioSemi order is plausible. Our posterior channels keep ~60 % of the tracking of the top-10 channels: less signal than at the best sites, but far from none.

In [ ]:
# Keep our 17 channels, preprocess to the common format (band-pass, 50 Hz, z-score per channel and recording)
AUD_REC = []
for subject, stim, eeg in tqdm(RECORDINGS, desc='Preprocessing public EEG', unit='recording'):
    x = bandpass_resample(eeg[CHANNEL_IDX], AUDEEG_SFREQ)
    AUD_REC.append((subject, stim, (x - x.mean(1, keepdims=True)) / x.std(1, keepdims=True)))
del RECORDINGS

## 4. Sound features (the same code for the speech stimuli and our videos)

Each 3-s window of sound (16 kHz, mono) becomes 150 steps of 20 ms:
- **envelope**: RMS per 20 ms, compressed with power 0.6 (loudness: the classic driver of EEG speech tracking);
- **log-mel**: Whisper's own 80-band log-mel (10-ms hop, two frames per step): when sounds start in which bands;
- **Whisper**: the hidden state of layer `WHISPER_LAYER` for the first 150 tokens (the window is padded with silence to Whisper's 30 s). The PCA to 64 dims is fitted on the training windows of the **speech** data and applied unchanged to our videos.

Feature extraction runs on all GPUs, one Whisper copy each. The speech features are cached in `AUDEEG_ROOT`.

In [ ]:
def stimulus_audio(name):
    """Original audio of one speech stimulus, mono float32 at AUDIO_RATE ('audio' and 'fs' in .npz or .npz.gz)."""
    hits = list((AUDEEG_ROOT / 'stimuli').rglob(f'{name}.npz*'))
    assert hits, f'no audio file for stimulus {name}'
    raw = gzip.open(hits[0]).read() if hits[0].suffix == '.gz' else hits[0].read_bytes()
    with np.load(io.BytesIO(raw), allow_pickle=False) as z:
        audio, fs = z['audio'].astype(np.float64), int(z['fs'])
    audio = audio.mean(axis=1) if audio.ndim == 2 else audio                 # stereo -> mono
    g = gcd(AUDIO_RATE, fs)
    return resample_poly(audio, AUDIO_RATE // g, fs // g).astype(np.float32)


STIMULI = sorted({stim for _, stim, _ in AUD_REC})
with ThreadPoolExecutor(LOAD_THREADS) as pool:
    AUDIO = dict(zip(STIMULI, tqdm(pool.map(stimulus_audio, STIMULI), total=len(STIMULI), desc='Speech audio',
                                   unit='stimulus')))

# unique stimulus windows: (stimulus, k) covers [3k, 3k + 3) s; validation = the last VAL_FRACTION of each stimulus
win_len = int(WINDOW_S * AUDIO_RATE)
WINDOWS, WIN_OF = [], {}
for stim in STIMULI:
    for k in range(len(AUDIO[stim]) // win_len):
        WIN_OF[(stim, k)] = len(WINDOWS)
        WINDOWS.append((stim, k, (k + 1) * WINDOW_S > (1 - VAL_FRACTION) * len(AUDIO[stim]) / AUDIO_RATE))
WIN_IS_VAL = np.array([v for _, _, v in WINDOWS])
print(f'{len(WINDOWS)} unique 3-s speech windows ({WIN_IS_VAL.mean():.0%} validation)')

In [ ]:
FEAT_CFG = dict(whisper=WHISPER_CHECKPOINT, layer=WHISPER_LAYER, envelope_power=ENVELOPE_POWER, rate=RATE_HZ)
FEAT_CACHE = AUDEEG_ROOT / f'audio_features_{tag(FEAT_CFG)}.npz'
FRAME = AUDIO_RATE // RATE_HZ                            # 320 samples = 20 ms
WHISPER = {f'cuda:{g}': (WhisperFeatureExtractor.from_pretrained(WHISPER_CHECKPOINT),          # one copy per GPU, loaded
                         WhisperModel.from_pretrained(WHISPER_CHECKPOINT).encoder.eval().to(f'cuda:{g}'))   # in turn
           for g in range(N_GPUS)}                                                             # (not thread-safe)


@torch.inference_mode()
def window_features(chunks, device):
    """List of 3-s audio arrays (16 kHz) -> envelope (B, 150), log-mel (B, 150, 80), Whisper (B, 150, 768), float16."""
    extractor, encoder = WHISPER[device]
    a = torch.as_tensor(np.stack(chunks), device=device)
    env = a.reshape(len(chunks), N_STEPS, FRAME).pow(2).mean(-1).sqrt().pow(ENVELOPE_POWER)
    mel = extractor(list(chunks), sampling_rate=AUDIO_RATE, return_tensors='pt', device=device)['input_features']
    mel = mel.to(device)
    hs = encoder(mel, output_hidden_states=True).hidden_states[WHISPER_LAYER][:, :N_STEPS]
    mel_steps = mel[:, :, :N_STEPS * MEL_STEPS_PER_STEP].reshape(len(chunks), mel.shape[1], N_STEPS, -1).mean(-1)
    return [t.half().cpu().numpy() for t in (env, mel_steps.transpose(1, 2), hs)]


def features_of(chunks, desc):
    """Features of many 3-s windows, spread over all GPUs (one shared text bar)."""
    batches = [chunks[i:i + FEATURE_BATCH] for i in range(0, len(chunks), FEATURE_BATCH)]
    out = run_parallel(lambda b, progress, device: window_features(b, device), batches, 1, desc, 'batch')
    return [np.concatenate([o[j] for o in out]) for j in range(3)]


if FEAT_CACHE.is_file():
    with np.load(FEAT_CACHE) as z:
        S_ENV, S_MEL, S_WHI = z['env'], z['mel'], z['whisper']
else:
    S_ENV, S_MEL, S_WHI = features_of([AUDIO[s][k * win_len:(k + 1) * win_len] for s, k, _ in WINDOWS],
                                      'Features: speech windows')
    np.savez(FEAT_CACHE, env=S_ENV, mel=S_MEL, whisper=S_WHI)

# PCA of the Whisper layer on the speech training windows, summed block by block (all tokens at once: ~40 GB float64)
BLOCK = 2048                         # technical: windows per GPU block
dim = S_WHI.shape[-1]
s1 = torch.zeros(dim, dtype=torch.float64, device='cuda:0')
s2 = torch.zeros(dim, dim, dtype=torch.float64, device='cuda:0')
n_tok = 0
train_windows = np.flatnonzero(~WIN_IS_VAL)
for i in tqdm(range(0, len(train_windows), BLOCK), desc='Whisper PCA', unit='block'):
    x = torch.as_tensor(S_WHI[train_windows[i:i + BLOCK]].reshape(-1, dim), dtype=torch.float64, device='cuda:0')
    s1 += x.sum(0)
    s2 += x.T @ x
    n_tok += len(x)
PCA_MEAN = s1 / n_tok
w, V = torch.linalg.eigh(s2 / n_tok - torch.outer(PCA_MEAN, PCA_MEAN))
PCA_BASIS = V[:, w.argsort(descending=True)[:WHISPER_PCA]]
print(f'Whisper PCA: {WHISPER_PCA} components keep {w.sort(descending=True).values[:WHISPER_PCA].sum() / w.sum():.0%} '
      'of the variance')


def combine(env, mel, whi):
    """(B, 150) + (B, 150, 80) + (B, 150, 768) -> float32 (B, 150, 1 + 80 + WHISPER_PCA), block by block."""
    w = np.concatenate([((torch.as_tensor(whi[i:i + BLOCK], dtype=torch.float64, device='cuda:0') - PCA_MEAN)
                         @ PCA_BASIS).float().cpu().numpy() for i in range(0, len(whi), BLOCK)])
    return np.concatenate([env[..., None].astype(np.float32), mel.astype(np.float32), w], axis=-1)


AUD_FEATS = combine(S_ENV, S_MEL, S_WHI)
mu, sd = AUD_FEATS[~WIN_IS_VAL].mean((0, 1)), AUD_FEATS[~WIN_IS_VAL].std((0, 1)) + 1e-6
AUD_FEATS = ((AUD_FEATS - mu) / sd).astype(np.float16)              # z-score with the speech training windows
N_FEATURES = AUD_FEATS.shape[-1]

# EEG windows of the public data: every recording cut at the same 3-s grid as its stimulus
eeg_list, subj_list, win_list = [], [], []
for subject, stim, x in AUD_REC:
    for k in range(x.shape[1] // N_STEPS):
        if (stim, k) in WIN_OF:
            eeg_list.append(x[:, k * N_STEPS:(k + 1) * N_STEPS])
            subj_list.append(AUD_SUBJECTS.index(subject))
            win_list.append(WIN_OF[(stim, k)])
AUD_EEG = np.stack(eeg_list).astype(np.float16)                     # (EEG windows, 17, 150)
AUD_SUBJ, AUD_WIN = np.array(subj_list), np.array(win_list)
del eeg_list, AUD_REC
print(f'speech feature bank {AUD_FEATS.shape}; {len(AUD_EEG)} EEG windows from {len(AUD_SUBJECTS)} subjects '
      f'({len(AUD_EEG) / len(set(win_list)):.1f} subjects per window on average)')

## 5. Our data in the same format

- **EEG**: the same band-pass/resampling function (500 → 50 Hz). Mean response and channel scale are removed later, per fold, with the fitting recordings only.
- **Sound**: the 3 s from video onset, decoded with PyAV. A video without an audio track gets silence. Features of all 1,102 videos with the same function and the same Whisper PCA; saved raw to `features/` for `04`.

In [ ]:
def load_training_eeg(sub):
    """Video IDs and training recordings (n, 17, 150) of one subject, band-passed, 50 Hz."""
    with np.load(DATA_ROOT / 'trn' / f'sub-{sub:02d}_trn.npz') as z:
        keys = sorted(z.files, key=int)
        ids, eeg = np.array([int(k) for k in keys]), np.stack([z[k] for k in keys])
    n, c, _ = eeg.shape
    return ids, bandpass_resample(eeg.reshape(n * c, -1), EEG_SFREQ).reshape(n, c, -1)   # all recordings in one call


def decode_audio(path):
    """Mono float32 audio at AUDIO_RATE, exactly WINDOW_S long (silence if the video has no audio track)."""
    n = int(round(WINDOW_S * AUDIO_RATE))
    with av.open(str(path), metadata_errors='ignore') as container:
        if not container.streams.audio:
            return np.zeros(n, np.float32)
        resampler = av.AudioResampler(format='flt', layout='mono', rate=AUDIO_RATE)
        chunks = []
        for f in container.decode(container.streams.audio[0]):
            chunks += [r.to_ndarray().reshape(-1) for r in resampler.resample(f)]
        chunks += [r.to_ndarray().reshape(-1) for r in resampler.resample(None)]
    audio = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros(0, np.float32)
    return np.pad(audio[:n], (0, max(0, n - len(audio))))


OURS = {sub: load_training_eeg(sub) for sub in tqdm(SUBJECTS, desc='Loading our training EEG', unit='subject')}
VIDEO_PATHS = {int(p.stem): p for folder in ['stimuli_train', 'stimuli_test']
               for p in (DATA_ROOT / folder).glob('*.mp4') if p.stem.isdigit()}
ALL_VIDEO_IDS = np.array(sorted(VIDEO_PATHS))
assert np.array_equal(ALL_VIDEO_IDS, np.arange(1, 1103)), 'feature rows must be videos 1..1102'
ROW_OF = np.full(ALL_VIDEO_IDS.max() + 1, -1)
ROW_OF[ALL_VIDEO_IDS] = np.arange(len(ALL_VIDEO_IDS))
with ThreadPoolExecutor(LOAD_THREADS) as pool:
    video_audio = list(tqdm(pool.map(lambda v: decode_audio(VIDEO_PATHS[v]), ALL_VIDEO_IDS), total=len(ALL_VIDEO_IDS),
                            desc='Video audio', unit='video'))
VID_FEATS = combine(*features_of(video_audio, 'Features: our videos'))   # (1102, 150, 145), z-scored per fold later
SILENT = np.array([not np.any(a) for a in video_audio])                  # no audio track or digital silence
np.save(OUT_DIR / 'features' / 'video_sound_features.npy', VID_FEATS)
np.save(OUT_DIR / 'features' / 'video_is_silent.npy', SILENT)
(OUT_DIR / 'features' / 'metadata.json').write_text(json.dumps(dict(
    rows='video IDs 1..1102', grid='3 s from video onset, 50 Hz (20-ms steps)',
    columns={'0': f'envelope (RMS per 20 ms, power {ENVELOPE_POWER})', '1-80': 'Whisper log-mel, 80 bands',
             f'81-{N_FEATURES - 1}': f'{WHISPER_CHECKPOINT} encoder layer {WHISPER_LAYER}, PCA {WHISPER_PCA} '
                                     '(fitted on the public speech training windows)'},
    scaling='none (raw); fit the scaling on training videos only', silent_videos=ALL_VIDEO_IDS[SILENT].tolist()),
    indent=1))
print(f'our sound features {VID_FEATS.shape}; {SILENT.sum()} silent videos:', ALL_VIDEO_IDS[SILENT].tolist())

## 6. Model and training loop

- **Stimulus encoder**: causal, so the EEG at time *t* is compared with sound up to *t*; its 1.2-s receptive field lets it learn the latency. The residual connections in the conv blocks are our choice (the handoff does not say).
- **EEG encoder**: per-subject plus shared spatial projection, depthwise temporal convolution; optionally two transformer layers over the 150 time steps, whose output starts at zero, so the model starts as the convolution-only encoder.
- **Score**: each latent channel is z-scored over time, so the score is the mean correlation of the 32 latent time courses. When ranking, the candidates are centred across the candidates (CORTIVA).
- **Batches**: 16 stimuli (speech windows or videos) plus **all** their recordings across subjects; all recordings of a stimulus are positives (multi-positive InfoNCE).

In [ ]:
class CausalBlock(nn.Module):
    def __init__(self, dilation):
        super().__init__()
        self.pad = (KERNEL - 1) * dilation
        self.conv = nn.Conv1d(HIDDEN, 2 * HIDDEN, KERNEL, dilation=dilation)
        self.norm = nn.GroupNorm(GROUP_NORM_GROUPS, HIDDEN)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):                                # (B, HIDDEN, T)
        y = F.glu(self.conv(F.pad(x, (self.pad, 0))), dim=1)
        return x + self.drop(self.norm(y))


class DualEncoder(nn.Module):
    def __init__(self, n_features, n_slots, eeg_encoder):
        super().__init__()
        self.stim_in = nn.Sequential(nn.Dropout(INPUT_DROPOUT), nn.Linear(n_features, HIDDEN))
        self.stim_blocks = nn.Sequential(*[CausalBlock(d) for d in DILATIONS])
        self.stim_out = nn.Conv1d(HIDDEN, LATENT, 1)
        self.subject_proj = nn.Parameter(torch.zeros(n_slots, LATENT, len(CHANNELS)))   # per subject, starts at 0
        self.shared_proj = nn.Linear(len(CHANNELS), LATENT, bias=False)
        self.eeg_conv = nn.Conv1d(LATENT, LATENT, EEG_KERNEL, padding=EEG_KERNEL // 2, groups=LATENT)
        self.use_tf = eeg_encoder == 'cortiva+transformer'
        if self.use_tf:
            self.tf_in = nn.Linear(LATENT, TF_DIM)
            self.tf_pos = nn.Parameter(0.02 * torch.randn(1, N_STEPS, TF_DIM))
            layer = nn.TransformerEncoderLayer(TF_DIM, TF_HEADS, 2 * TF_DIM, TF_DROPOUT, activation='gelu',
                                               batch_first=True, norm_first=True)
            self.tf = nn.TransformerEncoder(layer, TF_LAYERS, enable_nested_tensor=False)
            self.tf_out = nn.Linear(TF_DIM, LATENT)
            nn.init.zeros_(self.tf_out.weight)
            nn.init.zeros_(self.tf_out.bias)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / TEMPERATURE_INIT)))

    @staticmethod
    def zscore(z):                                       # per latent channel over time, then flattened
        z = (z - z.mean(-1, keepdim=True)) / (z.std(-1, keepdim=True) + 1e-6)
        return z.flatten(1) / math.sqrt(z.shape[1] * z.shape[2])

    def stimulus(self, feats):                           # (B, 150, K) -> (B, LATENT * 150)
        h = self.stim_in(feats).transpose(1, 2)
        return self.zscore(self.stim_out(self.stim_blocks(h)))

    def eeg(self, x, slot):                              # (B, 17, 150) -> (B, LATENT * 150)
        w = self.shared_proj.weight + self.subject_proj[slot]                     # (B, LATENT, 17)
        z = self.eeg_conv(torch.bmm(w, x))
        if self.use_tf:
            h = self.tf(self.tf_in(z.transpose(1, 2)) + self.tf_pos)
            z = z + self.tf_out(h).transpose(1, 2)
        return self.zscore(z)


INIT_LOCK = threading.Lock()         # torch.manual_seed is global: create models one at a time (reproducible weights)


def new_model(n_slots, eeg_encoder, seed, device):
    with INIT_LOCK:
        torch.manual_seed(seed)
        model = DualEncoder(N_FEATURES, n_slots, eeg_encoder)
    return model.to(device)


def info_nce(z_eeg, z_stim, target, logit_scale):
    """Symmetric multi-positive InfoNCE: each EEG window -> its stimulus among the batch stimuli, and each stimulus ->
    all of its recordings."""
    logits = logit_scale.exp().clamp(max=100) * z_eeg @ z_stim.T                # (EEG, stimuli)
    l_eeg = F.cross_entropy(logits, target)
    pos = F.one_hot(target, z_stim.shape[0]).float()
    l_stim = -(logits.log_softmax(0) * pos).sum(0) / pos.sum(0).clamp(min=1)
    return (l_eeg + l_stim[pos.sum(0) > 0].mean()) / 2


@torch.no_grad()
def scores(model, X, slot, feats):
    """Similarity of EEG windows X (n, 17, 150) to candidate stimuli feats (m, 150, K); candidates centred."""
    model.eval()
    s = model.stimulus(feats)
    s = s - s.mean(0, keepdim=True)
    return torch.cat([model.eeg(X[i:i + 4096], slot[i:i + 4096]) for i in range(0, len(X), 4096)]) @ s.T


def ranks(S, true_cols):
    """Rank of the true candidate per row (GPU; used for early stopping only)."""
    return (S > S[torch.arange(len(S), device=S.device), true_cols][:, None]).sum(1) + 1


def train(model, X, slot, stim_of, feats, epochs, seed, val_fn, patience, params=None, lr=LR, progress=None, name=None):
    """Batches of STIMULI_PER_BATCH stimuli with all their recordings (stim_of: feature row of each recording). Keeps
    the best epoch of val_fn and stops after `patience` epochs without improvement. params: trained parameters
    (default: all). With progress: reports every step to the shared bar."""
    gen = np.random.default_rng(seed)
    by_stim = {}
    for i, s in enumerate(stim_of.tolist()):
        by_stim.setdefault(s, []).append(i)
    stims = np.array(sorted(by_stim))
    steps = math.ceil(len(stims) / STIMULI_PER_BATCH)
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY,
                            fused=True)
    if progress is not None:
        progress.expect(epochs * steps)
    best, best_state, best_epoch, wait, done = -math.inf, None, 0, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        order = gen.permutation(stims)
        for b in range(steps):
            if STOP.is_set():
                raise RuntimeError('stopped')
            batch = order[b * STIMULI_PER_BATCH:(b + 1) * STIMULI_PER_BATCH]
            rows = np.concatenate([by_stim[s] for s in batch])
            local = {s: j for j, s in enumerate(batch.tolist())}
            target = torch.as_tensor([local[s] for s in stim_of[rows].tolist()], device=X.device)
            rows = torch.as_tensor(rows, device=X.device)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                z_stim = model.stimulus(feats[torch.as_tensor(batch, device=X.device)].float())
                z_eeg = model.eeg(X[rows].float(), slot[rows])
            loss = info_nce(z_eeg.float(), z_stim.float(), target, model.logit_scale)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            done += 1
            if progress is not None:
                progress.tick()
        score = val_fn(model)
        if progress is not None:
            progress.note(name, f'ep {epoch} val {score:.3f} best {max(best, score):.3f}')
        if score > best:
            best, best_epoch, wait = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    if progress is not None:
        progress.tick(epochs * steps - done)             # steps skipped by early stopping
    model.load_state_dict(best_state)
    return dict(best_val=best, best_epoch=best_epoch, epochs=epoch)

## 7. Pretraining on the speech EEG (2 encoders × 4 seeds, in parallel on the GPUs)

Validation is the same task as ours: each held-out EEG window is ranked against **102 candidate windows** (groups of held-out speech windows); chance top-5 4.9 %. A finished seed is loaded from its checkpoint, not retrained.

In [ ]:
PRE_CFG = dict(features=FEAT_CFG, n_features=N_FEATURES, channels=CHANNELS, band=BAND_HZ, rate=RATE_HZ,
               hidden=HIDDEN, latent=LATENT, dilations=DILATIONS, eeg_kernel=EEG_KERNEL, tf=(TF_DIM, TF_LAYERS, TF_HEADS),
               lr=LR, wd=WEIGHT_DECAY, batch=STIMULI_PER_BATCH, epochs=PRETRAIN_MAX_EPOCHS, patience=PRETRAIN_PATIENCE,
               subjects=len(AUD_SUBJECTS))
CKPT_DIR = OUT_ROOT / 'pretraining' / tag(PRE_CFG)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
N_SLOTS = len(AUD_SUBJECTS) + len(SUBJECTS)              # subject slots: public subjects first, then our six
OUR_SLOT = {sub: len(AUD_SUBJECTS) + i for i, sub in enumerate(SUBJECTS)}
ARCHITECTURE = dict(n_features=N_FEATURES, n_slots=N_SLOTS, our_slot=OUR_SLOT, n_channels=len(CHANNELS),
                    n_steps=N_STEPS, hidden=HIDDEN, latent=LATENT, kernel=KERNEL, dilations=DILATIONS,
                    input_dropout=INPUT_DROPOUT, dropout=DROPOUT, group_norm_groups=GROUP_NORM_GROUPS,
                    eeg_kernel=EEG_KERNEL, tf_dim=TF_DIM, tf_layers=TF_LAYERS, tf_heads=TF_HEADS, tf_dropout=TF_DROPOUT,
                    band_hz=BAND_HZ, rate_hz=RATE_HZ, eeg_sfreq=EEG_SFREQ)
FT_CFG = dict(ft_lr=FT_LR, ft_epochs=FT_MAX_EPOCHS, ft_patience=FT_PATIENCE, inner=INNER_FRACTION, inner_seed=INNER_SEED,
              lr=LR, wd=WEIGHT_DECAY, batch=STIMULI_PER_BATCH, candidate_centring=True, seed_ensemble='row z-score, mean')
(OUT_DIR / 'config.json').write_text(json.dumps(dict(
    run=RUN_ID, notebook='pretrained_transformers/abgabe/02_train_audio_transformer.ipynb', architecture=ARCHITECTURE,
    channels=CHANNELS, seeds=SEEDS, pretraining=PRE_CFG, finetuning=FT_CFG, settings=SETTINGS,
    versions=dict(python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
                  numpy=np.__version__)), indent=1, default=str))

IS_VAL_EEG = WIN_IS_VAL[AUD_WIN]
VAL_WINDOWS = np.flatnonzero(WIN_IS_VAL)
perm = np.random.default_rng(0).permutation(VAL_WINDOWS)
VAL_GROUPS = [np.sort(g) for g in perm[:len(perm) // N_CANDIDATES * N_CANDIDATES].reshape(-1, N_CANDIDATES)]


def pretrain(job, progress, device):
    encoder, seed = job
    ckpt = CKPT_DIR / f'{encoder}_seed{seed}.pt'
    meta = ckpt.with_suffix('.json')
    if ckpt.is_file() and meta.is_file():
        return json.loads(meta.read_text())
    t = lambda a, dtype=None: torch.as_tensor(a, dtype=dtype, device=device)
    X, slot, feats = t(AUD_EEG), t(AUD_SUBJ, torch.long), t(AUD_FEATS)
    tr = np.flatnonzero(~IS_VAL_EEG)
    val_rows = {w: np.flatnonzero(AUD_WIN == w) for w in VAL_WINDOWS}

    def val_fn(model):               # top-5 among 102 held-out speech windows, as on our test set
        hits = n = 0
        for g in VAL_GROUPS:
            rows = np.concatenate([val_rows[w] for w in g])
            S = scores(model, X[t(rows)].float(), slot[t(rows)], feats[t(g)].float())
            hits += int((ranks(S, t(np.searchsorted(g, AUD_WIN[rows]))) <= 5).sum())
            n += len(rows)
        return hits / n

    model = new_model(N_SLOTS, encoder, seed, device)
    info = train(model, X[t(tr)], slot[t(tr)], AUD_WIN[tr], feats, PRETRAIN_MAX_EPOCHS, seed, val_fn,
                 PRETRAIN_PATIENCE, progress=progress, name=f'{encoder} s{seed}')
    torch.save(model.state_dict(), ckpt)
    meta.write_text(json.dumps(dict(encoder=encoder, seed=seed, **info), indent=1))
    return json.loads(meta.read_text())


PRETRAIN = run_parallel(pretrain, [(e, s) for e in EEG_ENCODERS for s in SEEDS], 1, 'Pretraining on speech EEG', 'step',
                        count_items=False)
PRE_TABLE = pd.DataFrame(PRETRAIN)
PRE_TABLE.to_csv(OUT_DIR / 'pretraining.csv', index=False)
print(PRE_TABLE.round(3).to_string(index=False), f'\nchance top-5 {5 / N_CANDIDATES:.3f}')

## 8. Validation of all variants (5 folds × 4 seeds)

Per fold, our EEG gets the mean response removed and is scaled per channel, and the video features are z-scored, all with statistics of the fitting recordings and videos. `zero-shot`, `frozen` and `finetune` start from the speech checkpoint of the same encoder and seed; our subjects start with only the shared spatial projection (their own projections start at zero). The trained variants stop early on the inner 10 % of the fitting videos, as in `01`.

In [ ]:
def normalise(eeg, stats, device):
    mean, scale = stats
    return torch.as_tensor((eeg - mean) / scale, dtype=torch.float32, device=device)


def our_inputs(fit_videos, device, query_videos=None):
    """Fitting recordings of all six subjects and optional query recordings, normalised per subject with its fitting
    recordings (minus the mean response, divided by one std per channel). Also returns these statistics."""
    X, slot, vid, queries, stats = [], [], [], {}, {}
    for sub in SUBJECTS:
        ids, eeg = OURS[sub]
        fit = np.isin(ids, fit_videos)
        mean = eeg[fit].mean(0, keepdims=True)                                   # mean response (1, 17, 150)
        stats[sub] = (mean, (eeg[fit] - mean).std(axis=(0, 2), keepdims=True))   # and one std per channel
        X.append(normalise(eeg[fit], stats[sub], device))
        vid.append(ids[fit])
        slot.append(torch.full((int(fit.sum()),), OUR_SLOT[sub], device=device))
        if query_videos is not None:
            q = np.isin(ids, query_videos)
            queries[sub] = (normalise(eeg[q], stats[sub], device), ids[q])
    return torch.cat(X), torch.cat(slot), np.concatenate(vid), queries, stats


def fit_ours(setting, seed, fit_videos, device, query_videos=None):
    """Train one setting (variant, EEG encoder) on fit_videos minus an inner early-stopping set. Returns the model, the
    query recordings, the normalised features of all videos, the normalisation statistics and the training summary."""
    variant, encoder = setting
    X, slot, vid, queries, eeg_stats = our_inputs(fit_videos, device, query_videos)
    fv = VID_FEATS[ROW_OF[fit_videos]]
    feat_stats = (fv.mean((0, 1)), fv.std((0, 1)) + 1e-6)                        # fitting videos only
    feats = torch.as_tensor((VID_FEATS - feat_stats[0]) / feat_stats[1], dtype=torch.float32, device=device)
    model = new_model(N_SLOTS, encoder, seed, device)
    if variant != 'scratch':
        model.load_state_dict(torch.load(CKPT_DIR / f'{encoder}_seed{seed}.pt', map_location=device, weights_only=True))
    info = dict(best_epoch=0)
    if variant != 'zero-shot':
        inner = np.sort(np.random.default_rng(INNER_SEED).choice(fit_videos, round(INNER_FRACTION * len(fit_videos)),
                                                                 replace=False))
        is_inner = np.isin(vid, inner)
        m = torch.as_tensor(is_inner, device=device)
        inner_cand = feats[torch.as_tensor(ROW_OF[inner], device=device)]
        inner_true = torch.as_tensor(np.searchsorted(inner, vid[is_inner]), device=device)

        def val_fn(model):           # higher = better: minus the mean rank of the inner recordings among the inner videos
            return -float(ranks(scores(model, X[m], slot[m], inner_cand), inner_true).float().mean())

        info = train(model, X[~m], slot[~m], ROW_OF[vid[~is_inner]], feats, FT_MAX_EPOCHS, seed, val_fn, FT_PATIENCE,
                     params=[model.subject_proj] if variant == 'frozen' else None,
                     lr=FT_LR if variant == 'finetune' else LR)
    return model, queries, feats, dict(eeg=eeg_stats, features=feat_stats), info


def similarity(model, x, sub, cand):
    return scores(model, x, torch.full((len(x),), OUR_SLOT[sub], device=x.device), cand).cpu().numpy()


def validation_job(job, progress, device):
    setting, f, seed = job
    fold = FOLDS[f]
    model, queries, feats, _, info = fit_ours(setting, seed, np.setdiff1d(TRAIN_VIDEO_IDS, fold), device, fold)
    cand = feats[torch.as_tensor(ROW_OF[fold], device=device)]
    return dict(setting=setting, fold=f, seed=seed, best_epoch=info['best_epoch'],
                sims={sub: similarity(model, x, sub, cand) for sub, (x, _) in queries.items()},
                truth={sub: np.searchsorted(fold, ids) for sub, (_, ids) in queries.items()})


jobs = [(s, f, seed) for s in SETTINGS for f in range(N_FOLDS) for seed in SEEDS]
RUNS = run_parallel(validation_job, jobs, FT_JOBS_PER_GPU, 'Validation runs')

### Validation results

Per seed and seed ensemble (row z-score per seed, then the mean over the 4 seeds, as CORTIVA), mean over subjects and folds. Chance: top-1 0.010, top-5 0.049.

In [ ]:
NAMES = ['variant', 'encoder']
per_seed, grouped = [], {}
for r in RUNS:
    for sub in SUBJECTS:
        S, truth = r['sims'][sub], r['truth'][sub]
        per_seed.append(dict(zip(NAMES, r['setting']), fold=r['fold'], seed=r['seed'], subject=sub,
                             best_epoch=r['best_epoch'], **metrics(S, truth)))
        grouped.setdefault((r['setting'], r['fold'], sub), []).append((S, truth))
ENSEMBLE_SIMS = {key: (np.mean([row_z(S) for S, _ in v], axis=0), v[0][1]) for key, v in grouped.items()}
PER_SEED = pd.DataFrame(per_seed)
ENSEMBLE = pd.DataFrame([dict(zip(NAMES, s), fold=f, subject=sub, **metrics(S, truth))
                         for (s, f, sub), (S, truth) in ENSEMBLE_SIMS.items()])
SUMMARY = (ENSEMBLE.groupby(NAMES)[['top1', 'top5', 'mean_rank']].mean()
           .join(PER_SEED.groupby(NAMES)[['top5', 'best_epoch']].mean().rename(columns={'top5': 'top5_per_seed'})))
for name, table in [('per_seed', PER_SEED), ('seed_ensemble', ENSEMBLE), ('summary', SUMMARY.reset_index())]:
    table.to_csv(OUT_DIR / 'validation' / f'{name}.csv', index=False)
for setting in SETTINGS:             # seed-ensemble similarities, e.g. to fit a fusion with other models on validation
    np.savez(OUT_DIR / 'validation' / 'similarity' / f'{slug(setting)}.npz',
             **{f'fold{f}_sub{sub:02d}{part}': a for (s, f, sub), pair in ENSEMBLE_SIMS.items() if s == setting
                for part, a in zip(['', '_truth'], pair)})

labels = [' | '.join(k) for k in SUMMARY.index]
per_sub = ENSEMBLE.groupby(NAMES + ['subject'])['top5'].mean().unstack('subject').loc[SUMMARY.index]
colors = {v: f'C{i}' for i, v in enumerate(VARIANTS)}
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(labels)), SUMMARY.top5, color=[colors[k[0]] for k in SUMMARY.index], alpha=0.7)
for i, vals in enumerate(per_sub.values):
    ax.plot(np.full(len(vals), i), vals, 'k.', ms=3)
ax.axhline(5 / N_CANDIDATES, color='0.4', ls='--', lw=1, label='chance')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('validation top-5 (seed ensemble)')
ax.set_title('Sound only: does speech-EEG pretraining help? (colour = variant, dots = subjects)', fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / 'figures' / 'validation_variants.png', dpi=130, bbox_inches='tight')
plt.show()
SUMMARY.round(3)

**How to read it.** `finetune` above `scratch`: speech EEG teaches something that transfers to our videos. `zero-shot` above chance: the speech model works on our recordings without adaptation. `cortiva+transformer` above `cortiva`, but only after pretraining: pretraining makes a larger EEG encoder usable.

**Reference result** (same pipeline in `pretrained_transformers/06`, runs of 2026-10-18; a rerun differs slightly), seed ensemble:

| setting | top-1 | top-5 |
|---|---|---|
| `scratch`, `cortiva` | 0.137 | 0.331 |
| `scratch`, `cortiva+transformer` | 0.136 | 0.330 |
| `finetune`, `cortiva+transformer` | 0.134 | 0.327 |
| `frozen`, `cortiva+transformer` | 0.049 | 0.149 |
| `zero-shot`, `cortiva+transformer` | 0.016 | 0.064 |

- **Sound alone identifies the videos well:** top-5 0.33 on validation, ~1.4× the video transformer of `01` (0.229). Sound events are sharp in time and evoke time-locked responses over the full 3 s, while CLIP describes a whole clip with one static vector.
- **Pretraining on speech EEG does not help** (`finetune` ≈ `scratch`), and `zero-shot` is close to chance. Likely reasons: Flemish speech differs from the sound of our videos (music, noise, effects), speech tracking is weaker at our posterior channels (~60 % of the best channels), and our ~5,650 recordings already suffice for this small, nearly linear EEG encoder.
- **The transformer layers add nothing** (0.330 vs 0.331): attention over time brings no gain over the 180-ms convolution, with or without pretraining.
- This is a negative result for transfer learning, unlike for vision (`01`): what transfers depends on whether the pretraining stimuli and electrode sites match the target task.

## 9. Final models: every setting trained on all 1,000 training videos

As in `01`: each setting is trained on all training videos (10 % inner set for early stopping), once per seed. Saved per setting: `models/<setting>.pt` (4 state dicts, each subject's EEG normalisation, the feature scaling of the training videos) and `models/<setting>.json` (setting, best epochs, validation scores). `models/chosen.json` names the setting with the best validation top-5; `04` uses it for the submission. The test EEG is not read here.

In [ ]:
def final_job(job, progress, device):
    setting, seed = job
    model, _, _, stats, info = fit_ours(setting, seed, TRAIN_VIDEO_IDS, device)
    return dict(state={k: v.detach().cpu() for k, v in model.state_dict().items()}, best_epoch=info['best_epoch'],
                stats=stats)


final_jobs = [(s, seed) for s in SETTINGS for seed in SEEDS]
FINAL = dict(zip(final_jobs, run_parallel(final_job, final_jobs, FT_JOBS_PER_GPU, 'Final models')))
for setting in SETTINGS:
    runs = [FINAL[(setting, seed)] for seed in SEEDS]
    stats = runs[0]['stats']                                   # the same for all seeds (same recordings and videos)
    torch.save(dict(states={seed: r['state'] for seed, r in zip(SEEDS, runs)},
                    eeg_normalisation={sub: dict(mean=torch.as_tensor(m), scale=torch.as_tensor(s))
                                       for sub, (m, s) in stats['eeg'].items()},
                    feature_normalisation=dict(mean=torch.as_tensor(stats['features'][0]),
                                               std=torch.as_tensor(stats['features'][1]))),
               OUT_DIR / 'models' / f'{slug(setting)}.pt')
    (OUT_DIR / 'models' / f'{slug(setting)}.json').write_text(json.dumps(dict(
        name=slug(setting), **dict(zip(NAMES, setting)), seeds=SEEDS, best_epochs=[r['best_epoch'] for r in runs],
        validation=SUMMARY.loc[setting, ['top1', 'top5', 'mean_rank']].round(4).to_dict()), indent=1))

best = SUMMARY[SELECTION_METRIC].idxmax()
(OUT_DIR / 'models' / 'chosen.json').write_text(json.dumps(dict(
    name=slug(best), **dict(zip(NAMES, best)), rule=f'best validation {SELECTION_METRIC} (seed ensemble)',
    validation=SUMMARY.loc[best, ['top1', 'top5', 'mean_rank']].round(4).to_dict()), indent=1))
print(f'{len(SETTINGS)} final models saved in {OUT_DIR / "models"}\nchosen on validation: {slug(best)}')